In [1]:
import os
import time
import requests
import psycopg2
import subprocess
from datetime import datetime
from tqdm import tqdm
from dotenv import load_dotenv
from concurrent.futures import ThreadPoolExecutor, as_completed

# Cargar variables de entorno
load_dotenv()

# Mantener la Mac despierta (macOS)
caffeinate_process = subprocess.Popen(['caffeinate'])

# Configuración de conexión
dbname = os.getenv("POSTGRES_DB")
user = os.getenv("POSTGRES_USER")
password = os.getenv("POSTGRES_PASSWORD")
host = os.getenv("POSTGRES_HOST")
port = os.getenv("POSTGRES_PORT")
API_KEY = os.getenv("FMP_API_KEY")

# Conexión a PostgreSQL
conn = psycopg2.connect(
    dbname=dbname,
    user=user,
    password=password,
    host=host,
    port=port
)
cursor = conn.cursor()

# Obtener tickers desde la tabla de prueba (o cambiar a api_raw.active_tickers si vas a producción)
def obtener_tickers():
    try:
        cursor.execute("SELECT ticker FROM infraestructura.tickers_test_api_raw")
        return [t[0] for t in cursor.fetchall()]
    except Exception as e:
        print(f"Error al obtener tickers: {e}")
        return []

# Llamada a la API de FMP para ratios TTM
def obtener_ratios_ttm(ticker):
    try:
        url = f"https://financialmodelingprep.com/api/v3/ratios-ttm/{ticker}?apikey={API_KEY}"
        response = requests.get(url)

        if response.status_code != 200:
            print(f"Error {response.status_code} para {ticker}")
            return None

        data = response.json()
        if not data:
            print(f"No data para {ticker}")
            return None

        r = data[0]
        hoy = datetime.now().date()

        return {
            'ticker': ticker,
            'fecha_de_consulta': hoy,
            'GrossProfitMarginTTM': r.get('grossProfitMarginTTM'),
            'OperatingProfitMarginTTM': r.get('operatingProfitMarginTTM'),
            'NetProfitMarginTTM': r.get('netProfitMarginTTM'),
            'ReturnOnEquityTTM': r.get('returnOnEquityTTM'),
            'PriceEarningsRatioTTM': r.get('priceEarningsRatioTTM'),
            'PriceEarningsToGrowthRatioTTM': r.get('priceEarningsToGrowthRatioTTM'),
            'PriceToSalesRatioTTM': r.get('priceToSalesRatioTTM'),
            'CurrentRatioTTM': r.get('currentRatioTTM'),
            'QuickRatioTTM': r.get('quickRatioTTM'),
            'CashFlowToDebtRatioTTM': r.get('cashFlowToDebtRatioTTM'),
            'FreeCashFlowPerShareTTM': r.get('freeCashFlowPerShareTTM'),
            'DebtRatioTTM': r.get('debtRatioTTM'),
            'DebtEquityRatioTTM': r.get('debtEquityRatioTTM'),
            'InterestCoverageTTM': r.get('interestCoverageTTM'),
            'PayoutRatioTTM': r.get('payoutRatioTTM')
        }

    except Exception as e:
        print(f"Error con {ticker}: {e}")
        return None

# Insertar en la base de datos
def insertar_ratios(datos):
    try:
        query = """
            INSERT INTO api_raw.ratios_ttm (
                ticker, fecha_de_consulta, GrossProfitMarginTTM, OperatingProfitMarginTTM,
                NetProfitMarginTTM, ReturnOnEquityTTM, PriceEarningsRatioTTM, 
                PriceEarningsToGrowthRatioTTM, PriceToSalesRatioTTM, CurrentRatioTTM,
                QuickRatioTTM, CashFlowToDebtRatioTTM, FreeCashFlowPerShareTTM,
                DebtRatioTTM, DebtEquityRatioTTM, InterestCoverageTTM, PayoutRatioTTM
            )
            VALUES (
                %(ticker)s, %(fecha_de_consulta)s, %(GrossProfitMarginTTM)s, %(OperatingProfitMarginTTM)s,
                %(NetProfitMarginTTM)s, %(ReturnOnEquityTTM)s, %(PriceEarningsRatioTTM)s,
                %(PriceEarningsToGrowthRatioTTM)s, %(PriceToSalesRatioTTM)s, %(CurrentRatioTTM)s,
                %(QuickRatioTTM)s, %(CashFlowToDebtRatioTTM)s, %(FreeCashFlowPerShareTTM)s,
                %(DebtRatioTTM)s, %(DebtEquityRatioTTM)s, %(InterestCoverageTTM)s, %(PayoutRatioTTM)s
            )
            ON CONFLICT (ticker, fecha_de_consulta) DO NOTHING
        """
        cursor.executemany(query, datos)
        conn.commit()
        print(f"✅ {len(datos)} ratios TTM insertados.")
    except Exception as e:
        print(f"❌ Error insertando datos: {e}")
        conn.rollback()

# Proceso principal
def ejecutar_proceso():
    inicio = time.time()
    tickers = obtener_tickers()

    if not tickers:
        print("⚠️ No se encontraron tickers.")
        return

    lote = 300
    with ThreadPoolExecutor(max_workers=5) as executor:
        for i in range(0, len(tickers), lote):
            t_lote = tickers[i:i + lote]
            futures = {executor.submit(obtener_ratios_ttm, t): t for t in t_lote}
            resultados = []

            for future in tqdm(as_completed(futures), total=len(t_lote), desc=f"Lote {i//lote + 1}"):
                r = future.result()
                if r:
                    resultados.append(r)

            if resultados:
                insertar_ratios(resultados)

            if i + lote < len(tickers):
                print("⏳ Esperando 30s para evitar rate limit...")
                time.sleep(30)

    fin = time.time()
    caffeinate_process.terminate()
    cursor.close()
    conn.close()
    print(f"⏱️ Tiempo total: {round(fin - inicio, 2)} segundos.")

# Ejecutar
if __name__ == "__main__":
    ejecutar_proceso()


Lote 1: 100%|█████████████████████████████████████| 5/5 [00:00<00:00,  6.55it/s]

✅ 5 ratios TTM insertados.
⏱️ Tiempo total: 0.81 segundos.
